# Topic-Hard Split Experiment (Notebook First)

This notebook upgrades the baseline clustering pipeline to:

1. Build topic vectors with **TF-IDF -> TruncatedSVD -> L2 normalization**.
2. Evaluate multiple `k` values (`[20, 30, 40, 50, 60, 80]`) using quality and balance diagnostics.
3. Choose a more usable clustering.
4. Simulate whole-cluster assignment to train/dev/test with a greedy objective.
5. Print split quality summaries **without writing files by default**.

Use this notebook to validate the method before creating an actual split file.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


In [ ]:
# Paths + experiment config
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data.topic_hard_split import (
    _DEFAULT_BALANCE_BAND,
    _DEFAULT_K_VALUES,
    _RANDOM_STATE,
    add_usable_score,
    assign_clusters_to_splits,
    build_split_json,
    build_topic_vectors,
    choose_usable_k,
    evaluate_k_values,
    load_cleaned_data,
    plot_cluster_diagnostics,
    print_final_summary,
    save_split,
    select_cluster_examples,
    summarize_clusters,
    top_terms_by_cluster,
)

CLEANED_PATH = PROJECT_ROOT / "artifacts" / "data" / "cleaned.jsonl"
STANDARD_SPLIT_PATH = PROJECT_ROOT / "artifacts" / "splits" / "standard.json"
TOPIC_HARD_SPLIT_PATH = PROJECT_ROOT / "artifacts" / "splits" / "topic_hard.json"

RANDOM_STATE = _RANDOM_STATE
K_VALUES = list(_DEFAULT_K_VALUES)
BALANCE_BAND = _DEFAULT_BALANCE_BAND
TARGET_RATIOS = {"train": 0.8, "dev": 0.1, "test": 0.1}

# Safety switch: keep False during experimentation.
SAVE_SPLIT = True

df = load_cleaned_data(CLEANED_PATH)

print(f"Loaded {len(df):,} rows")
df.head()


In [ ]:
def format_summary_table_for_display(summary_table: pd.DataFrame):
    formatted = summary_table.copy()

    def _format_value(value: float) -> str:
        value = float(value)
        if value.is_integer():
            return f"{int(value):,}"
        return f"{value:,.1f}"

    formatted["Value"] = formatted["Value"].map(_format_value)
    return formatted.style.hide(axis="index")


def format_examples_table_for_display(example_table: pd.DataFrame):
    return example_table.style.hide(axis="index")


In [ ]:
x_tfidf, x_topic, vectorizer, reducer = build_topic_vectors(
    df["text"],
    max_features=20_000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    n_components=200,
    random_state=RANDOM_STATE,
)

print(f"TF-IDF shape: {x_tfidf.shape}")
print(f"Topic vector shape: {x_topic.shape}")

metrics_df, labels_by_k = evaluate_k_values(
    X_topic=x_topic,
    labels_true=df["label"],
    k_values=K_VALUES,
    random_state=RANDOM_STATE,
    silhouette_sample_size=5000,
)
metrics_df = add_usable_score(metrics_df)

metrics_df.sort_values("usable_score", ascending=False).reset_index(drop=True)


In [5]:
chosen_k = choose_usable_k(metrics_df)
print(f"Chosen k: {chosen_k}")

df_clustered = df.copy()
df_clustered["cluster"] = labels_by_k[chosen_k]

df_clustered[["id", "label", "cluster", "text"]].head()

Chosen k: 80


,id,label,cluster,text
0,sar_000001,1,72,thirtysomething scientists unveil doomsday clo...
1,sar_000002,0,55,dem rep. totally nails why congress is falling...
2,sar_000003,0,5,eat your veggies: 9 deliciously different recipes
3,sar_000004,1,5,inclement weather prevents liar from getting t...
4,sar_000005,1,5,mother comes pretty close to using word 'strea...


In [ ]:
summary_table_df, cluster_profile_df = summarize_clusters(
    df_clustered,
    balance_band=BALANCE_BAND,
)

print("Summary table for report/slide use")
display(format_summary_table_for_display(summary_table_df))


In [ ]:
top_terms_df = top_terms_by_cluster(
    X_tfidf=x_tfidf,
    cluster_labels=df_clustered["cluster"].to_numpy(),
    vectorizer=vectorizer,
    top_n=12,
)

cluster_examples_df = select_cluster_examples(cluster_profile_df, top_terms_df)

print("Example clusters for appendix or backup slides")
display(format_examples_table_for_display(cluster_examples_df))

plot_cluster_diagnostics(
    cluster_profile_df,
    balance_band=BALANCE_BAND,
)


In [ ]:
df_dryrun, cluster_to_split = assign_clusters_to_splits(
    df=df_clustered,
    split_ratios=TARGET_RATIOS,
    random_state=RANDOM_STATE,
)

print_final_summary(
    df_assigned=df_dryrun,
    cluster_to_split=cluster_to_split,
    target_ratios=TARGET_RATIOS,
)

split_sizes = df_dryrun["split"].value_counts().reindex(["train", "dev", "test"])
split_sizes


In [ ]:
# Optional write step (disabled by default).
if SAVE_SPLIT:
    split_obj = build_split_json(
        df_assigned=df_dryrun,
        cluster_to_split=cluster_to_split,
        metrics=metrics_df,
        chosen_k=chosen_k,
        split_ratios=TARGET_RATIOS,
        random_state=RANDOM_STATE,
    )

    output_path = save_split(split_obj, TOPIC_HARD_SPLIT_PATH)
    print(f"Saved split -> {output_path}")
else:
    print("SAVE_SPLIT=False: dry-run only, no split file written.")
